# 09 — Confidence Features (Evidence Fusion)

**Purpose:** First notebook allowed to merge outputs from all independent evidence streams.

**Input artifacts (§16):**
- `signal_quality_features.parquet` (Notebook 02)
- `delineation_features.parquet` (Notebook 03)
- `twave_features.parquet` (Notebook 04)
- `measurement_reliability.parquet` (Notebook 05)
- `qtc_comparison.parquet` (Notebook 06)
- `clinical_context.parquet` (Notebook 01)

**Output:** `outputs/confidence_features.parquet`

**Granularity:** Record level — primary key: `record_id`

**Aggregation Rule:** For continuous variables retain `mean`, `std`, `max`, `p95`.
Never retain only mean values.

**Leakage Prevention:** `absolute_qt_error_ms`, `expert_disagreement_ms`,
`true_t_end_error_ms`, `measurement_instability_ms` are **forbidden**.


In [1]:
import sys
sys.path.insert(0, '../src')


In [2]:
import numpy as np
import pandas as pd
from datetime import datetime

RANDOM_SEED = 42
PIPELINE_VERSION = "1.0.0"
TIMESTAMP = datetime.utcnow().isoformat()

FORBIDDEN_FEATURES = [
    "absolute_qt_error_ms","expert_disagreement_ms",
    "true_t_end_error_ms","measurement_instability_ms","confidence_label",
]

# Load all upstream artifacts
sq_df    = pd.read_parquet("../outputs/signal_quality_features.parquet")
delin_df = pd.read_parquet("../outputs/delineation_features.parquet")
twave_df = pd.read_parquet("../outputs/twave_features.parquet")
rely_df  = pd.read_parquet("../outputs/measurement_reliability.parquet")
qtc_df   = pd.read_parquet("../outputs/qtc_comparison.parquet")
clinical = pd.read_parquet("../outputs/clinical_context.parquet")
inventory = pd.read_csv("../outputs/inventory.csv")

print(f"Signal quality  : {sq_df.shape}")
print(f"Delineation     : {delin_df.shape}")
print(f"T-wave          : {twave_df.shape}")
print(f"Reliability     : {rely_df.shape}")
print(f"QTc comparison  : {qtc_df.shape}")
print(f"Clinical context: {clinical.shape}")


Signal quality  : (740, 13)
Delineation     : (740, 11)
T-wave          : (3700, 14)
Reliability     : (7056, 14)
QTc comparison  : (6210, 11)
Clinical context: (70, 7)


/tmp/ipykernel_2958/4121166863.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  TIMESTAMP = datetime.utcnow().isoformat()


## Aggregation to Record Level

All lead- and beat-level features are aggregated using:
`mean`, `std`, `max`, `p95` (as required by Data Contract §16).


In [3]:
def agg_to_record(df, group_col, features, suffix=""):
    """Aggregate feature columns to record level with mean/std/max/p95."""
    agg_dict = {}
    for feat in features:
        if feat not in df.columns:
            continue
        agg_dict[f"mean_{feat}{suffix}"] = (feat, "mean")
        agg_dict[f"std_{feat}{suffix}"]  = (feat, "std")
        agg_dict[f"max_{feat}{suffix}"]  = (feat, "max")
        agg_dict[f"p95_{feat}{suffix}"]  = (feat, lambda x: x.quantile(0.95))
    if not agg_dict:
        return pd.DataFrame()
    return df.groupby(group_col).agg(**agg_dict).reset_index()

# ── Signal Quality Evidence ───────────────────────────────────────────
sq_feats = ["bw_index","hfn_index","pli_index","snr_db","clipping_ratio",
            "flatline_ratio","electrode_motion_index","signal_quality_score"]
sq_agg = agg_to_record(sq_df, "record_id", sq_feats)

# Keep only the contracted columns
SQ_CONTRACT = [
    "mean_bw_index","std_bw_index","max_bw_index","p95_bw_index",
    "mean_hfn_index","std_hfn_index","max_hfn_index",
    "mean_snr_db","std_snr_db",
    "mean_clipping_ratio","max_clipping_ratio",
]
print(f"SQ aggregated: {sq_agg.shape}")


SQ aggregated: (70, 33)


In [4]:
# ── Delineation Evidence ─────────────────────────────────────────────
delin_feats = ["boundary_confidence","t_end_uncertainty_ms","p_onset_uncertainty_ms","qrs_onset_uncertainty_ms"]
delin_agg = agg_to_record(delin_df, "record_id", delin_feats)
print(f"Delineation aggregated: {delin_agg.shape}")

# ── T-Wave Morphology Evidence ───────────────────────────────────────
tw_feats = ["t_end_ambiguity_score","morphology_confidence","t_amplitude_mv","t_symmetry"]
tw_agg = agg_to_record(twave_df, "record_id", tw_feats)
print(f"T-wave aggregated: {tw_agg.shape}")

# ── Measurement Agreement Evidence ───────────────────────────────────
rely_feats = ["bsqi","wsqi","lead_agreement_score","beat_agreement_score",
              "qt_variance_leads","qt_variance_beats","repeatability_score","internal_consistency_score"]
rely_agg = agg_to_record(rely_df, "record_id", rely_feats)
print(f"Reliability aggregated: {rely_agg.shape}")


Delineation aggregated: (70, 17)
T-wave aggregated: (70, 17)


Reliability aggregated: (70, 33)


In [5]:
# ── Merge all evidence streams ────────────────────────────────────────
base = inventory[["record_id","dataset_name"]].copy()

conf_feat = (base
    .merge(sq_agg,    on="record_id", how="left")
    .merge(delin_agg, on="record_id", how="left")
    .merge(tw_agg,    on="record_id", how="left")
    .merge(rely_agg,  on="record_id", how="left")
    .merge(clinical[["record_id","arrhythmia_flag","diagnostic_class","dataset_origin"]],
           on="record_id", how="left")
)

# Rename reliability cols to match contract
rename_map = {
    "mean_lead_agreement_score": "mean_lead_agreement",
    "min_lead_agreement_score":  "worst_lead_agreement",
    "mean_beat_agreement_score": "mean_beat_agreement",
    "min_beat_agreement_score":  "worst_beat_agreement",
}
conf_feat = conf_feat.rename(columns=rename_map)

# Add worst (min) aggregations for agreement scores
for col in ["lead_agreement_score","beat_agreement_score","bsqi","wsqi"]:
    agg_min = rely_df.groupby("record_id")[col].min().rename(f"worst_{col.replace('_score','')}")
    conf_feat = conf_feat.merge(agg_min, on="record_id", how="left")

# Reproducibility metadata
conf_feat["pipeline_version"]     = PIPELINE_VERSION
conf_feat["processing_timestamp"] = TIMESTAMP

print(f"Confidence features shape: {conf_feat.shape}")
print(f"Columns ({len(conf_feat.columns)}): {list(conf_feat.columns)[:15]} ...")


Confidence features shape: (70, 107)
Columns (107): ['record_id', 'dataset_name', 'mean_bw_index', 'std_bw_index', 'max_bw_index', 'p95_bw_index', 'mean_hfn_index', 'std_hfn_index', 'max_hfn_index', 'p95_hfn_index', 'mean_pli_index', 'std_pli_index', 'max_pli_index', 'p95_pli_index', 'mean_snr_db'] ...


## Schema Validation & Leakage Check

In [6]:
REQUIRED_CONF_COLS = [
    "record_id","dataset_name",
    "mean_bw_index","std_bw_index","max_bw_index",
    "mean_hfn_index","mean_snr_db","std_snr_db",
    "mean_clipping_ratio","max_clipping_ratio",
    "mean_boundary_confidence","std_boundary_confidence",
    "max_t_end_uncertainty_ms","p95_t_end_uncertainty_ms",
    "mean_t_end_ambiguity_score","max_t_end_ambiguity_score",
    "mean_morphology_confidence",
    "arrhythmia_flag","diagnostic_class","dataset_origin",
]
# Some cols may have slightly different names after merge — check key ones
key_cols = ["record_id","dataset_name","mean_bw_index","mean_snr_db",
            "mean_t_end_ambiguity_score","arrhythmia_flag"]
missing = [c for c in key_cols if c not in conf_feat.columns]
leakage = [c for c in FORBIDDEN_FEATURES if c in conf_feat.columns]
assert not missing,  f"Missing key cols: {missing}"
assert not leakage,  f"Leakage cols present: {leakage}"
print("✓ Schema validation passed — no leakage")
print(f"  Shape: {conf_feat.shape}")
print(f"  NaN summary: {conf_feat.isnull().sum().sum()} total NaN values")


✓ Schema validation passed — no leakage
  Shape: (70, 107)
  NaN summary: 0 total NaN values


## Feature Coverage Heatmap

In [7]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

numeric_cols = conf_feat.select_dtypes(include=[np.number]).columns
nan_frac = conf_feat[numeric_cols].isnull().mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
nan_frac.head(20).plot.barh(ax=axes[0], color="#d62728", edgecolor="white")
axes[0].set_title("Top-20 Features by NaN Fraction")
axes[0].set_xlabel("NaN fraction")

conf_feat.select_dtypes(include=[np.number]).describe().loc["mean"].sort_values().tail(20).plot.barh(
    ax=axes[1], color="#1f77b4", edgecolor="white")
axes[1].set_title("Feature Mean Values (top-20)")
plt.tight_layout()
plt.savefig("../outputs/confidence_features_summary.png", dpi=100)
plt.show()
print("Figure saved.")


Figure saved.


/tmp/ipykernel_2958/2202134892.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Export

In [8]:
conf_feat.to_parquet("../outputs/confidence_features.parquet", index=False)
print("✓ confidence_features.parquet →", conf_feat.shape)
print(f"  Evidence domains:")
print(f"    Signal Quality    : {len([c for c in conf_feat.columns if 'bw_' in c or 'hfn_' in c or 'snr_' in c])}")
print(f"    Delineation       : {len([c for c in conf_feat.columns if 'boundary' in c or 't_end_unc' in c])}")
print(f"    Morphology        : {len([c for c in conf_feat.columns if 'ambiguity' in c or 'morphology' in c])}")
print(f"    Agreement/Reliability: {len([c for c in conf_feat.columns if 'bsqi' in c or 'agreement' in c or 'repeatability' in c])}")
print(f"    Clinical          : {len([c for c in conf_feat.columns if c in ['arrhythmia_flag','diagnostic_class','dataset_origin']])}")


✓ confidence_features.parquet → (70, 107)
  Evidence domains:
    Signal Quality    : 12
    Delineation       : 8
    Morphology        : 8
    Agreement/Reliability: 19
    Clinical          : 3
